# nixtla-scaffold feature tour

This notebook is a progressive demo of the major public forecasting surfaces, from the smallest useful forecast to hierarchy, experiments, refresh/ledger operations, and release gates. It stays no-bloat by running only small synthetic data with baseline models where possible, then using command/artifact maps for heavier optional workflows.

Use it as the guided walkthrough for humans and agents: it shows what to run, what artifacts to inspect, and how the package was made agent-friendly without hiding forecasting judgment behind magic.

## Forecasting caveats

A statistical forecast projects historical patterns forward. It is not a plan, target, or guarantee. Treat the point forecast, scenario overlays, external plans, and official locks as separate things. Use the generated trust, interval, residual, hierarchy, and data-quality artifacts before using any forecast for planning decisions.

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:  # Allows simple script-based smoke tests without Jupyter installed.
    def display(obj):
        print(obj)

# Make the notebook work from the repo root or from examples/feature_tour.
start = Path.cwd().resolve()
candidates = [start, *start.parents]
ROOT = next(
    path
    for path in candidates
    if (path / "pyproject.toml").exists() and (path / "src" / "nixtla_scaffold").exists()
)
src = ROOT / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from nixtla_scaffold import (  # noqa: E402
    CustomModelSpec,
    DriverEvent,
    KnownFutureRegressor,
    TransformSpec,
    aggregate_hierarchy_frame,
    compare_models,
    forecast_spec_preset,
    preset_catalog,
    profile_dataset,
    run_experiment,
    run_forecast,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

OUTPUT_ROOT = Path(os.environ.get("NIXTLA_FEATURE_TOUR_OUTPUT", ROOT / "runs" / "feature_tour"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT

## 1. Progression map: simple to complex

The default path stays intentionally small: profile data, run a forecast, inspect artifacts. Move right only when the decision needs it.

In [ ]:
progression_map = pd.DataFrame(
    [
        (1, "Data contract", "Use canonical unique_id/ds/y or map columns with CLI flags.", "Every run"),
        (2, "Profile", "Check frequency, gaps, short history, zeros, negatives, duplicates.", "Before modeling"),
        (3, "Quick forecast", "Baseline ladder, intervals when possible, trust artifacts.", "First read"),
        (4, "Finance forecast", "Events, target transforms, normalization, units, strict horizon controls.", "Planning baseline"),
        (5, "Model controls", "Policies, allowlists, compare_models, weighted ensemble, custom challengers.", "When champion choice matters"),
        (6, "Drivers", "Known-future regressor contracts, leakage/future-value audit, optional ML training.", "When business drivers exist"),
        (7, "Hierarchy", "Leaf-to-parent nodes, bottom-up/top-down/both reconciliation evidence.", "When totals must tie"),
        (8, "Experiment", "Bounded variants with an autoresearch next-iteration block.", "When improving one hypothesis at a time"),
        (9, "Operations", "setup, ingest, refresh, compare/score external forecasts, ledger.", "When repeated or MCP-backed"),
        (10, "Quality gates", "scenario-lab, workbench-qa, release-gates, guide/skill docs.", "Before broad use"),
    ],
    columns=["step", "area", "what it teaches", "when to use"],
)
progression_map

## 2. Public surface checklist

This is the coverage checklist for the demo. Some rows run below; heavier operational rows are shown as commands so the notebook stays fast.

In [ ]:
surface_checklist = pd.DataFrame(
    [
        ("setup", "Create an agent-ready workspace with config, intake questions, and agent_brief.md.", "Command shown; not run because setup is workspace-specific."),
        ("profile", "Profile canonical or mapped data before modeling.", "Run below through profile_dataset."),
        ("forecast", "Run model tournament, backtests, intervals, trust, reports, Streamlit.", "Run below twice: quick and standard."),
        ("compare-models", "Advisory leaderboard over the normal tournament.", "Run below through compare_models."),
        ("experiment", "Bounded variant comparison plus autoresearch next iteration.", "Run below through run_experiment."),
        ("refresh", "Reuse a prior manifest/spec against updated actuals.", "Command shown in operations table."),
        ("ingest", "Convert MCP/Kusto/DAX/SQL/Excel exports into unique_id/ds/y with provenance.", "Command shown in operations table."),
        ("hierarchy", "Aggregate hierarchy columns into Total/parent/leaf forecast nodes.", "Run below through aggregate_hierarchy_frame."),
        ("explain", "Read an existing model_card.md/model explanation.", "Artifact/command shown."),
        ("report", "Regenerate report.html/base64/Streamlit from an existing run.", "Artifact/command shown."),
        ("compare", "Align scaffold forecast to an external finance forecast.", "Command shown."),
        ("score-external", "Score cutoff-labeled external forecasts against actuals.", "Command shown."),
        ("scenario-lab", "Synthetic numeric validation scenarios.", "Command shown in quality gates."),
        ("workbench-qa", "Golden generated-dashboard/artifact QA.", "Command shown in quality gates."),
        ("release-gates", "Package/workbench release-readiness gates.", "Command shown in quality gates."),
        ("ledger", "Register versions, official locks, actuals, adjustments, and Power BI exports.", "Command shown in operations table."),
        ("guide", "Search embedded Nixtla/FPPy guidance.", "Command shown in quality gates."),
    ],
    columns=["surface", "purpose", "coverage in this notebook"],
)
surface_checklist

## 3. How the package is agent-friendly

Agent-friendly does not mean autonomous overreach. It means the package gives agents stable contracts, small bounded choices, and audit artifacts that make it hard to silently make a bad forecast.

In [ ]:
agent_friendly_design = pd.DataFrame(
    [
        ("Canonical data contract", "Everything reduces to unique_id/ds/y, with explicit id/time/target mapping flags.", "Agents do not guess schemas after the first mapping."),
        ("Intake/setup artifacts", "setup writes forecast_setup.yaml, questions.json, and agent_brief.md.", "The next agent can resume with the same assumptions."),
        ("Presets before knobs", "quick, standard, strict, hierarchy encode safe defaults; finance remains a legacy alias.", "Agents start simple and only override when the decision needs it."),
        ("Deterministic artifact layout", "Root stays readable; detail CSVs live in appendix/ and raw traces in audit/.", "Agents know exactly where to look."),
        ("Manifest and diagnostics", "manifest.json, diagnostics.json, model_card.md, and LLM context preserve run intent and warnings.", "Handoffs cite evidence instead of hallucinating confidence."),
        ("Fail-closed driver audit", "Known-future regressors need future values, known-as-of timing, and leakage checks.", "Drivers are not automatically trained because a column exists."),
        ("Advisory experiments", "compare_models and experiment do not overwrite forecast.csv champions.", "Iteration is safe and reversible."),
        ("Autoresearch loop", "experiment_llm_context.json contains one hypothesis, metric, budget, executor, and keep rule.", "Agents can improve forecasts one bounded test at a time."),
        ("Local workbench", "OPEN_ME_FIRST.html, report.html, and streamlit_app.py are generated artifacts.", "Review does not depend on a cloud service."),
        ("Quality gates", "workbench-qa and release-gates validate generated artifacts and dashboard behavior.", "Agent-built changes have a repeatable acceptance path."),
    ],
    columns=["design choice", "how it works", "why agents benefit"],
)
agent_friendly_design

## 4. Presets: fewer decisions at the start

Presets make the first run understandable. Override individual fields later when the decision needs stricter validation, hierarchy, model-family control, or feature policies.

In [ ]:
pd.DataFrame(preset_catalog())

## 5. Build demo data

The main demo frame uses `unique_id`, `ds`, `y`, a positive `price_factor` for normalization, and a `seats_plan` driver candidate. A second leaf-level frame demonstrates hierarchy aggregation.

In [ ]:
def build_finance_panel() -> pd.DataFrame:
    dates = pd.date_range("2024-01-31", periods=18, freq="ME")
    rows = []
    for unique_id, base, growth, seasonal_amp, starting_seats in [
        ("Core", 240_000, 0.018, 0.05, 1_800),
        ("Expansion", 95_000, 0.033, 0.09, 650),
    ]:
        for idx, ds in enumerate(dates):
            season = 1 + seasonal_amp * np.sin(2 * np.pi * (idx % 12) / 12)
            price_factor = 1 + 0.01 * (idx // 6)
            seats_plan = starting_seats + idx * (22 if unique_id == "Core" else 17)
            y = base * ((1 + growth) ** idx) * season * price_factor
            rows.append(
                {
                    "unique_id": unique_id,
                    "ds": ds,
                    "y": round(y, 2),
                    "price_factor": price_factor,
                    "seats_plan": seats_plan,
                }
            )
    return pd.DataFrame(rows)


def build_hierarchy_leaf_data() -> pd.DataFrame:
    dates = pd.date_range("2024-01-31", periods=18, freq="ME")
    rows = []
    for region, product, base, growth in [
        ("AMER", "Core", 120_000, 0.018),
        ("AMER", "Expansion", 55_000, 0.035),
        ("EMEA", "Core", 80_000, 0.015),
        ("EMEA", "Expansion", 38_000, 0.031),
    ]:
        for idx, ds in enumerate(dates):
            quarter_bump = 1.04 if ds.month in {3, 6, 9, 12} else 1.0
            rows.append(
                {
                    "region": region,
                    "product": product,
                    "ds": ds,
                    "y": round(base * ((1 + growth) ** idx) * quarter_bump, 2),
                }
            )
    return pd.DataFrame(rows)


finance_df = build_finance_panel()
hierarchy_leaf_df = build_hierarchy_leaf_data()
finance_df.head()

## 6. Profile before forecasting

Profiling is the first guardrail. It catches the boring problems that make forecasts misleading: irregular time index, short history, duplicates, nulls, zeros, and negatives.

In [ ]:
profile_spec = forecast_spec_preset("standard", horizon=4, freq="ME", model_policy="baseline")
profile = profile_dataset(finance_df[["unique_id", "ds", "y"]], profile_spec)
profile_summary = {key: value for key, value in profile.to_dict().items() if key != "series"}
display(pd.DataFrame([profile_summary]))
pd.DataFrame(profile.to_dict()["series"])

## 7. Smallest useful forecast

Start with a fast baseline run. This already writes the same durable artifact layout as larger runs, so the next agent or analyst can inspect it without rerunning anything.

In [ ]:
quick_spec = forecast_spec_preset("quick", horizon=3, freq="ME", model_policy="baseline")
quick_run = run_forecast(finance_df[["unique_id", "ds", "y"]], quick_spec)
quick_output = quick_run.to_directory(OUTPUT_ROOT / "quick_baseline")
print(f"Wrote quick run to: {quick_output}")
quick_trust = pd.read_csv(quick_output / "appendix" / "trust_summary.csv")
quick_series_features = pd.read_csv(quick_output / "appendix" / "series_features.csv")
display(quick_run.forecast.head(6))
display(quick_trust[[col for col in ["unique_id", "trust_level", "horizon_trust_state", "full_horizon_claim_allowed", "next_action"] if col in quick_trust.columns]])
quick_series_features[[col for col in ["unique_id", "history_depth_bucket", "zero_fraction", "coefficient_of_variation", "recommended_experiment_next_step"] if col in quick_series_features.columns]]

## 8. Finance forecast with intervals, events, normalization, and driver audit

This run adds common finance needs without changing the core contract: 80/95 interval requests, a price-factor normalization audit, an event overlay, and a known-future regressor contract. The statistical `yhat`, event-adjusted scenario, and driver audit stay separate.

In [ ]:
last_ds = finance_df["ds"].max()
future_dates = pd.date_range(last_ds + pd.offsets.MonthEnd(1), periods=4, freq="ME")
future_rows = []
for unique_id, start_value, step in [("Core", 2_225, 24), ("Expansion", 970, 19)]:
    for idx, ds in enumerate(future_dates):
        future_rows.append(
            {
                "unique_id": unique_id,
                "ds": ds,
                "seats_plan": start_value + idx * step,
                "known_as_of": last_ds,
            }
        )
future_regressor_file = OUTPUT_ROOT / "seats_plan_future.csv"
pd.DataFrame(future_rows).to_csv(future_regressor_file, index=False)
future_regressor_file

In [ ]:
finance_spec = forecast_spec_preset(
    "standard",
    horizon=4,
    freq="ME",
    model_policy="baseline",
    transform=TransformSpec(normalization_factor_col="price_factor", normalization_label="Demo price index"),
    events=(
        DriverEvent(
            name="Sales capacity ramp",
            start="2025-07-31",
            end="2025-10-31",
            effect="multiplicative",
            magnitude=0.06,
            confidence=0.75,
        ),
    ),
    regressors=(
        KnownFutureRegressor(
            name="Seats plan",
            value_col="seats_plan",
            availability="plan",
            mode="model_candidate",
            future_file=str(future_regressor_file),
            source_system="demo_planning_sheet",
            owner="finance",
            notes="Demo future plan; audited but not trained unless train_known_future_regressors=True.",
        ),
    ),
)

finance_run = run_forecast(finance_df, finance_spec)
finance_output = finance_run.to_directory(OUTPUT_ROOT / "finance_driver_audit")
print(f"Wrote finance run to: {finance_output}")
display(finance_run.forecast.head(8))
finance_run.model_selection[[col for col in ["unique_id", "selected_model", "rmse", "mae", "wape", "selection_horizon", "cv_windows"] if col in finance_run.model_selection.columns]]

## 9. Review artifacts and copy-friendly summaries

Open `OPEN_ME_FIRST.html` first for the curated launch page. For decisions, inspect trust, intervals, model audit, driver audit, transform audit, and the Streamlit workbench. Streamlit also has a selectable copy/paste wide summary with raw, thousands, millions, currency, currency-thousands, and currency-millions formats; the same idea is shown here in a notebook table. The Model investigation and Assumptions & Drivers tabs include a lightweight Nixtla-style MLForecast explainability guide: start with `model_explainability.csv`, then only reproduce `models_`, `preprocess`, `SaveFeatures`, or SHAP offline if a deeper investigation is worth the dependency/artifact cost. The Assumptions & Drivers tab also discovers optional regressor/driver/STL image artifacts from the run root, appendix, audit, or experiments folders so visual driver evidence can travel with the run without adding new default artifacts.

In [ ]:
artifact_map = pd.DataFrame(
    [
        ("Curated landing page", finance_output / "OPEN_ME_FIRST.html"),
        ("Compact workbook", finance_output / "output" / "forecast_review.xlsx"),
        ("Static report", finance_output / "report.html"),
        ("Streamlit app", finance_output / "streamlit_app.py"),
        ("Trust summary", finance_output / "appendix" / "trust_summary.csv"),
        ("Series features", finance_output / "appendix" / "series_features.csv"),
        ("Model audit", finance_output / "appendix" / "model_audit.csv"),
        ("Prediction interval diagnostics", finance_output / "appendix" / "interval_diagnostics.csv"),
        ("Target transform audit", finance_output / "audit" / "target_transform_audit.csv"),
        ("Scenario assumptions", finance_output / "appendix" / "scenario_assumptions.csv"),
        ("Scenario forecast", finance_output / "appendix" / "scenario_forecast.csv"),
        ("Known-future regressors", finance_output / "appendix" / "known_future_regressors.csv"),
        ("Driver availability audit", finance_output / "appendix" / "driver_availability_audit.csv"),
        ("Selected forecast", finance_output / "forecast.csv"),
        ("All future model rows", finance_output / "appendix" / "forecast_long.csv"),
        ("Rolling-origin rows", finance_output / "appendix" / "backtest_long.csv"),
    ],
    columns=["artifact", "path"],
)
artifact_map

In [ ]:
forecast_wide = finance_run.forecast.pivot_table(index="unique_id", columns="ds", values="yhat", aggfunc="first")
forecast_wide.columns = [pd.Timestamp(col).strftime("%Y-%m") for col in forecast_wide.columns]
formatted_wide = forecast_wide.apply(lambda col: col.map(lambda value: f"${value / 1_000_000:,.2f}M"))

trust = pd.read_csv(finance_output / "appendix" / "trust_summary.csv")
driver_audit = pd.read_csv(finance_output / "appendix" / "driver_availability_audit.csv")
interval_cols = [col for col in finance_run.forecast.columns if "lo" in col or "hi" in col]
print(f"Interval columns in forecast output: {interval_cols}")
display(formatted_wide)
display(trust[[col for col in ["unique_id", "trust_level", "horizon_trust_state", "full_horizon_claim_allowed", "caveats", "next_action"] if col in trust.columns]])
driver_audit[[col for col in ["name", "audit_status", "modeling_decision", "required_future_rows", "missing_future_rows", "audit_message"] if col in driver_audit.columns]]

In [ ]:
diagnostics_payload = json.loads((finance_output / "diagnostics.json").read_text(encoding="utf-8"))
manifest_payload = json.loads((finance_output / "manifest.json").read_text(encoding="utf-8"))
agent_handoff = {
    "diagnostics_top_level_keys": sorted(diagnostics_payload.keys()),
    "manifest_output_count": len(manifest_payload.get("outputs", {})),
    "llm_safe_reminders": diagnostics_payload.get("next_steps", [])[:5],
}
agent_handoff

## 10. Model controls: policies, allowlists, compare_models, and custom challengers

`compare_models` gives analysts and agents a readable leaderboard over the same underlying tournament. It is advisory and does not override the official selected forecast. Use model policies, model allowlist controls, and custom challengers only when the decision needs that extra control.

In [ ]:
leaderboard = compare_models(
    finance_df[["unique_id", "ds", "y"]],
    spec=forecast_spec_preset("quick", horizon=3, freq="ME", model_policy="baseline"),
    output_dir=OUTPUT_ROOT / "compare_models",
)
leaderboard.head(12)

In [ ]:
model_control_reference = pd.DataFrame(
    [
        ("Fast baselines", "--model-policy baseline", "Runs always-on sanity models."),
        ("Classical only", "--model-policy statsforecast", "Uses StatsForecast candidates when installed."),
        ("ML only", "--model-policy mlforecast", "Uses optional MLForecast lag/date features."),
        ("All families", "--model-policy all", "Explicit broader tournament; still audit-gated."),
        ("Model allowlist", "--model arima --model 'arima mstl'", "Alias-friendly allowlist; non-allowlisted candidates skip."),
        ("Literal favorites", "--model-allowlist AutoARIMA MSTL_AutoARIMA --no-weighted-ensemble", "No derived WeightedEnsemble when the user wants only named models."),
        ("Rolling ML features", "--mlforecast-feature-policy rolling", "Small audited target-history rolling transforms; optional."),
        ("Known-future drivers", "--train-known-future-regressors", "Only audited model_candidate regressors can enter MLForecast."),
        ("Strict horizon", "--strict-cv-horizon --require-backtest", "No full-horizon validation, no champion claim."),
    ],
    columns=["control", "CLI shape", "guardrail"],
)
model_control_reference

In [ ]:
def last_observed_custom(history, horizon, freq, cutoff, levels, context):
    future_grid = pd.DataFrame(context["future_grid"])[["unique_id", "ds"]].copy()
    last_values = (
        history.sort_values("ds")
        .groupby("unique_id", as_index=False)
        .tail(1)[["unique_id", "y"]]
        .rename(columns={"y": "yhat"})
    )
    return future_grid.merge(last_values, on="unique_id", how="left")


custom_spec = forecast_spec_preset(
    "quick",
    horizon=2,
    freq="ME",
    model_policy="baseline",
    custom_models=(CustomModelSpec(name="Last observed challenger", callable=last_observed_custom),),
)
custom_run = run_forecast(finance_df.loc[finance_df["unique_id"].eq("Core"), ["unique_id", "ds", "y"]], custom_spec)
custom_output = custom_run.to_directory(OUTPUT_ROOT / "custom_challenger")
print(f"Wrote custom challenger run to: {custom_output}")
custom_run.custom_model_contracts[[col for col in ["model", "status", "invocation_count", "output_contract", "leakage_guard"] if col in custom_run.custom_model_contracts.columns]]

## 11. Hierarchy: leaf data to coherent planning totals

Hierarchy runs start from leaf rows, create Total/parent/child nodes, then reconcile when parent and child forecasts must tie. The `both` mode keeps bottom-up as the primary forecast and writes bottom-up vs top-down comparison evidence. If one child has less history than others, model selection remains per node: short children can fall back to simpler models, while parent/total nodes can still use their own available history. Review trust and reconciliation artifacts before using the coherent total.

In [ ]:
hierarchy_nodes = aggregate_hierarchy_frame(
    hierarchy_leaf_df,
    hierarchy_cols=("region", "product"),
    time_col="ds",
    target_col="y",
)
display(hierarchy_nodes.head(10))

hierarchy_spec = forecast_spec_preset(
    "hierarchy",
    horizon=2,
    freq="ME",
    model_policy="baseline",
    hierarchy_reconciliation="both",
)
hierarchy_run = run_forecast(hierarchy_nodes, hierarchy_spec)
hierarchy_output = hierarchy_run.to_directory(OUTPUT_ROOT / "hierarchy_both")
print(f"Wrote hierarchy run to: {hierarchy_output}")
display(hierarchy_run.hierarchy_reconciliation_comparison.head(12))
pd.read_csv(hierarchy_output / "appendix" / "hierarchy_reconciliation.csv").head(12)

## 12. Bounded experiments and autoresearch-style iteration

Experiments are intentionally small. They write normal child run folders plus recommendation markdown and `experiment_llm_context.json`. The autoresearch block is the important agent handoff: one hypothesis, one metric, one executor, one fixed budget, and an explicit keep/discard rule.

In [ ]:
experiment_result = run_experiment(
    finance_df,
    spec=finance_spec,
    output_dir=OUTPUT_ROOT / "experiment_events",
    variants=("baseline", "events"),
    max_variants=2,
)
print(f"Wrote experiment to: {experiment_result.output_dir}")
display(experiment_result.summary)
autoresearch_next = experiment_result.llm_context["recommendation"]["autoresearch_next_iteration"]
display(pd.DataFrame([autoresearch_next]))
print(experiment_result.recommendation_markdown)

## 13. Operational workflows after the first forecast

These are usually not run inside a fast demo notebook, but they are part of the product surface and complete the path from one-off analysis to repeatable forecasting.

In [ ]:
operations_commands = pd.DataFrame(
    [
        ("setup", r"uv run nixtla-scaffold setup --workspace runs\workspace --data-source kusto --preset standard --target-name revenue --time-col Month --horizon 6 --freq ME --exploration-mode", "Creates forecast_setup.yaml, questions.json, agent_brief.md, and query/template folders."),
        ("ingest", r"uv run nixtla-scaffold ingest --input query_result.json --source kusto --query-file source.kql --id-value Revenue --time-col Month --target-col Revenue --output runs\revenue_input.csv --forecast-output runs\revenue_forecast --preset standard --horizon 6", "Preserves MCP/query provenance while converting to unique_id/ds/y."),
        ("refresh", r"uv run nixtla-scaffold refresh --previous-run runs\revenue_forecast --input updated_actuals.csv --output runs\revenue_refresh", "Reuses prior manifest/spec, reruns model selection under that same policy, and writes deltas."),
        ("explain", r"uv run nixtla-scaffold explain --run runs\revenue_forecast", "Reads model_card.md/model explanation from a saved run."),
        ("report", r"uv run nixtla-scaffold report --run runs\revenue_forecast", "Regenerates report.html, report_base64.txt, and streamlit_app.py."),
        ("compare external", r"uv run nixtla-scaffold compare --run runs\revenue_forecast --external finance_plan.xlsx --format wide", "Triangulates scaffold yhat against a finance-owned plan/forecast."),
        ("score external", r"uv run nixtla-scaffold score-external --external external_cutoffs.csv --actuals actuals.csv --output runs\external_score", "Scores cutoff-labeled external forecasts without letting them leak into model selection."),
        ("ledger init/register", r"uv run nixtla-scaffold ledger init --ledger runs\forecast_ledger; uv run nixtla-scaffold ledger register --ledger runs\forecast_ledger --run runs\revenue_forecast --forecast-key Revenue --version-label demo", "Creates a refreshable version ledger."),
        ("ledger lock/actuals/export", r"uv run nixtla-scaffold ledger lock --ledger runs\forecast_ledger --forecast-key Revenue --version-id <id>; uv run nixtla-scaffold ledger export --ledger runs\forecast_ledger", "Marks official submissions, appends actuals/revisions, and exports Power BI-ready CSV mirrors."),
    ],
    columns=["workflow", "example command", "why it matters"],
)
operations_commands

## 14. Refreshability: rerun, compare, lock, monitor drift

Refreshability should feel boring: new actuals land, point `refresh` at the last run, and get a new run folder with the same setup plus explicit deltas. By default refresh **does not pin the old selected model**. It reuses the previous manifest spec, applies only explicit overrides, reruns the normal tournament on the updated data, and selects the best current champion under the same policy/allowlist. If the champion changes, `appendix/refresh_delta.csv` records a `model_selection_change` row.

That is the right default for model drift: models often get worse when the process changes, seasonality shifts, new products launch, or the old champion overfit an older regime. Governance should come from deltas, ledger locks, and post-actual scoring rather than silently freezing stale models.

In [ ]:
refreshability_reference = pd.DataFrame(
    [
        ("Default refresh", "Reuse previous manifest spec, rerun forecast/model tournament, choose the current champion.", "Best default for recurring monthly/weekly updates."),
        ("Keep the same candidate set", "Reuse the prior model_policy/model_allowlist, or pass --model / --model-allowlist explicitly.", "Use when stakeholders want controlled model governance."),
        ("Pin one prior champion", "Not a separate magic flag today; constrain the next run to that model with --model-allowlist if you truly want a champion freeze.", "Use sparingly; stale champions can hide drift."),
        ("Forecast movement", "Review refresh_delta.csv rows where delta_type == forecast_yhat_change.", "Answers what moved in the refreshed forecast."),
        ("Champion drift", "Review delta_type == model_selection_change plus model_audit/backtest metrics.", "Flags model flapping or a new best family."),
        ("Trust drift", "Review delta_type == trust_change and trust_summary.csv.", "Catches degraded readiness before a forecast is used for planning."),
        ("Official submission", "Use ledger lock/register/export to preserve what was sent to leadership and compare later refreshes.", "Separates official locks from exploratory refreshes."),
        ("When models get worse", "Score landed actuals, inspect residual/window metrics, shorten horizon, add events/drivers, or run a bounded experiment.", "Turns drift into an agent-friendly next iteration instead of manual guessing."),
    ],
    columns=["question", "answer", "why it matters"],
)
refreshability_reference

In [ ]:
refresh_operating_loop = pd.DataFrame(
    [
        (1, "New actuals arrive", "Append or replace the canonical unique_id/ds/y input from the source of truth."),
        (2, "Run refresh", r"uv run nixtla-scaffold refresh --previous-run runs\last --input updated_actuals.csv --output runs\next"),
        (3, "Review deltas", "Open refresh_manifest.json and appendix/refresh_delta.csv for spec, yhat, champion, and trust changes."),
        (4, "Decide governance", "If this is official, register/lock it in the forecast ledger; otherwise keep it exploratory."),
        (5, "Score after actuals land", "Use ledger exports or score-external to measure whether recent forecast error is widening."),
        (6, "Respond to drift", "Run experiment/autoresearch iteration when trust drops, champions flap, or errors worsen."),
    ],
    columns=["step", "action", "artifact or command"],
)
refresh_operating_loop

## 15. Quality gates, guide, and skill handoff

Use these before claiming generated workbench changes, release readiness, or agent documentation are done.

In [ ]:
quality_commands = pd.DataFrame(
    [
        ("scenario-lab", r"uv run nixtla-scaffold scenario-lab --count 20 --model-policy baseline --output runs\scenario_lab_fast", "Synthetic numeric scenarios for validity/accuracy/ease/explainability."),
        ("workbench-qa", r"uv run nixtla-scaffold workbench-qa --model-policy baseline --no-app-test --output runs\workbench_qa_fast", "Golden generated run/dashboard/artifact checks."),
        ("release-gates", r"uv run nixtla-scaffold release-gates --output runs\release_gates_fast --no-build --no-install-smoke --no-workbench-qa --no-live-streamlit --scenario-count 2", "Local release-readiness gates; run fuller gates before broad release."),
        ("guide", "uv run nixtla-scaffold guide hierarchy", "Embedded Nixtla/FPPy best-practice guidance for agents."),
        ("skill", r"skills\nixtla-forecast\SKILL.md", "Active agent instructions mirror the notebook, artifacts, regressors, hierarchy, experiments, and autoresearch loop."),
    ],
    columns=["surface", "example", "use"],
)
quality_commands

## 16. How to continue the demo

1. Open `OPEN_ME_FIRST.html` from the finance run.
2. Launch the generated Streamlit app from the run folder with `run_streamlit.ps1`.
3. In Streamlit, start in Forecast review, then try Model investigation MLForecast explainability, Prediction intervals, Hierarchy, Assumptions & Drivers regressor visual evidence, Feeder outputs, and the copy/paste wide summary controls.
4. Compare `quick_baseline`, `finance_driver_audit`, `compare_models`, `custom_challenger`, `hierarchy_both`, and `experiment_events` under the output root.
5. Use the refreshability section to decide when a refresh should freely reselect the best current champion versus when governance requires a constrained model allowlist.
6. Use the operational command map for setup/ingest/refresh/ledger/external-forecast workflows when you move beyond a one-off demo.

The goal is not to memorize every feature. The goal is to know which artifact or workflow answers each planning question, and to keep every agent iteration bounded, auditable, and reversible.

In [ ]:
pd.DataFrame(
    [
        ("Output root", OUTPUT_ROOT),
        ("Quick run", quick_output),
        ("Finance run", finance_output),
        ("Finance Streamlit launcher", finance_output / "run_streamlit.ps1"),
        ("Compare models run", OUTPUT_ROOT / "compare_models"),
        ("Custom challenger run", custom_output),
        ("Hierarchy run", hierarchy_output),
        ("Hierarchy Streamlit launcher", hierarchy_output / "run_streamlit.ps1"),
        ("Experiment recommendation", experiment_result.output_dir / "experiment_recommendation.md"),
        ("Experiment LLM context", experiment_result.output_dir / "experiment_llm_context.json"),
    ],
    columns=["item", "path"],
)